# Phase 3: Smart Inventory Planning

## Objective
Transform SKU-level demand forecasts into actionable inventory decisions by combining forecasted demand, current stock levels, supplier lead times, and incoming purchase orders.

## Scope
- **Demand Forecasting:** Use the existing XGBoost forecasts for October–December 2026.
- **Inventory Assessment:** Evaluate current stock levels, existing safety stock, reorder points, and days of cover.
- **Inventory Position:** Account for available stock and outstanding purchase orders.
- **Smart Safety Stock:** Analyze historical forecast bias and evaluate a conservative inventory buffer for the demo.
- **Replenishment Recommendations:** Identify SKUs requiring replenishment and estimate recommended order quantities.

## Expected Deliverable
An SKU-level inventory planning table containing:
- Forecasted Demand
- Current Stock and Inventory Position
- Supplier Lead Time
- Existing and Proposed Safety Stock
- Reorder Point
- Recommended Reorder Quantity
- Recommended Action

## Methodology
1. Validate inventory and purchase-order data.
2. Align the inventory snapshot with the demand forecasting period.
3. Estimate demand during the replenishment lead time.
4. Evaluate forecast bias and define a transparent inventory buffer.
5. Calculate replenishment requirements and generate explainable recommendations.

## Key Consideration
Forecast overprediction will be evaluated as a potential conservative planning signal, not automatically treated as statistically valid safety stock. The final recommendations will distinguish expected demand from the additional inventory buffer.

**Business Goal:** Reduce the risk of stockouts while avoiding unnecessary inventory accumulation.

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import warnings
import joblib

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

In [6]:
DATA_DIR = Path(r"D:\HVIA\task2\Sharnouby_Enterprise")

DATA_FILES = {
"inventory_history": {
"filename": "Inventory_History.csv",
"date_columns": ["Snapshot_Month"],
},
"sales_transactions": {
"filename": "Sales_Transactions.csv",
"date_columns": ["Order_Date", "Return_Date"],
},
"purchase_orders": {
"filename": "Purchase_Orders.csv",
"date_columns": ["Order_Date", "Promised_Date", "Actual_Receipt_Date"],
},
"product_master": {
"filename": "Product_Master.csv",
"date_columns": [],
},
}

dataframes = {}

for name, file_info in DATA_FILES.items():
 filename = file_info["filename"]
 date_columns = file_info["date_columns"]
 file_path = DATA_DIR / filename

 if file_path.exists():
    dataframes[name] = pd.read_csv(
        file_path,
        parse_dates=date_columns,
    )

    print(
        f"{filename}: Loaded successfully — "
        f"{dataframes[name].shape}"
    )
 else:
    print(f"{filename}: File not found at {file_path}")

Inventory_History.csv: Loaded successfully — (6372, 19)
Sales_Transactions.csv: Loaded successfully — (180000, 30)
Purchase_Orders.csv: Loaded successfully — (3600, 13)
Product_Master.csv: Loaded successfully — (59, 12)


In [7]:
KEY_COLUMNS = {
    "inventory_history": ["Snapshot_Month", "Location_ID", "SKU"],
    "sales_transactions": ["Row_ID", "Invoice_ID", "SKU"],
    "purchase_orders": ["PO_ID", "Material_ID", "Supplier_ID"],
    "product_master": ["SKU", "Product_ID"],
}

for name, columns in KEY_COLUMNS.items():
    df = dataframes[name]

    print(f"\n{'=' * 60}")
    print(f"Dataset: {name}")

    available_columns = [col for col in columns if col in df.columns]
    missing_columns = [col for col in columns if col not in df.columns]

    print(f"Available key columns: {available_columns}")
    print(f"Missing columns: {missing_columns}")

    if available_columns:
        print(
            "Duplicate key combinations:",
            df.duplicated(subset=available_columns).sum()
        )

        print("\nUnique values per key column:")
        print(df[available_columns].nunique(dropna=False))


Dataset: inventory_history
Available key columns: ['Snapshot_Month', 'Location_ID', 'SKU']
Missing columns: []
Duplicate key combinations: 0

Unique values per key column:
Snapshot_Month    36
Location_ID        3
SKU               59
dtype: int64

Dataset: sales_transactions
Available key columns: ['Row_ID', 'Invoice_ID', 'SKU']
Missing columns: []
Duplicate key combinations: 0

Unique values per key column:
Row_ID        180000
Invoice_ID    180000
SKU               59
dtype: int64

Dataset: purchase_orders
Available key columns: ['PO_ID', 'Material_ID', 'Supplier_ID']
Missing columns: []
Duplicate key combinations: 0

Unique values per key column:
PO_ID          3600
Material_ID      12
Supplier_ID      12
dtype: int64

Dataset: product_master
Available key columns: ['SKU', 'Product_ID']
Missing columns: []
Duplicate key combinations: 0

Unique values per key column:
SKU           59
Product_ID    59
dtype: int64


In [8]:
inventory_history = dataframes["inventory_history"]

latest_month = inventory_history["Snapshot_Month"].max()

latest_inventory = inventory_history.loc[
    inventory_history["Snapshot_Month"] == latest_month
].copy()

print(f"Latest Snapshot: {latest_month}")
print(f"Shape: {latest_inventory.shape}")
print(f"Unique SKUs: {latest_inventory['SKU'].nunique()}")
print(f"Unique Locations: {latest_inventory['Location_ID'].nunique()}")

Latest Snapshot: 2026-09-01 00:00:00
Shape: (177, 19)
Unique SKUs: 59
Unique Locations: 3


In [9]:
sku_inventory = (
    latest_inventory
    .groupby(["SKU", "Product_ID", "Product_Name"], as_index=False)
    .agg(
        Total_Closing_Stock=("Closing_Stock", "sum"),
        Total_Safety_Stock=("Safety_Stock_Units", "sum"),
        Total_Reorder_Point=("Reorder_Point_Units", "sum"),
        Avg_Daily_Sales_90D=("Avg_Daily_Sales_90D", "sum"),
        Supplier_Lead_Time_Days=("Supplier_Lead_Time_Days", "max"),
        Total_Stockout_Days=("Stockout_Days_In_Month", "sum"),
        Locations_Count=("Location_ID", "nunique"),
    )
)

print(f"Inventory Shape: {sku_inventory.shape}")
print(f"Unique SKUs: {sku_inventory['SKU'].nunique()}")

display(sku_inventory.head())

Inventory Shape: (59, 10)
Unique SKUs: 59


,SKU,Product_ID,Product_Name,Total_Closing_Stock,Total_Safety_Stock,Total_Reorder_Point,Avg_Daily_Sales_90D,Supplier_Lead_Time_Days,Total_Stockout_Days,Locations_Count
0,26115-1,P006,Summer children's boys-Sonic,288,349,885,38.85,21,0,3
1,26115-2,P005,Summer children's boys-Black and orange,588,714,1812,55.70,21,0,3
2,26115-3,P007,Summer children's boys-Black and turquoise,719,88,223,26.95,5,0,3
3,26115-4,P008,Summer children's boys-SpongeBob,1061,657,1667,100.66,14,0,3
4,26115-5,P003,Summer children's girls-Hello kitty Rose & tur...,583,169,427,49.23,7,0,3


In [13]:
MODEL_PATH = Path("saved_models/final_sku_xgboost_model.joblib")

if MODEL_PATH.exists():
    model_artifact = joblib.load(MODEL_PATH)

    print("Model loaded successfully.")
    print(f"File size: {MODEL_PATH.stat().st_size / 1024:.2f} KB")
    print(f"Model type: {model_artifact['model_type']}")
    print(f"Target: {model_artifact['target']}")
    print(f"Training period: {model_artifact['data_start']} to {model_artifact['data_end']}")
    print(f"Forecast months: {model_artifact['forecast_months']}")
    print(f"Features: {model_artifact['features']}")

else:
    print(f"Model file not found: {MODEL_PATH.resolve()}")

Model loaded successfully.
File size: 345.15 KB
Model type: XGBRegressor
Target: Target_Quantity
Training period: 2023-09 to 2026-09
Forecast months: ['2026-10', '2026-11', '2026-12']
Features: ['SKU', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_6', 'Lag_12', 'Rolling_Mean_3', 'Rolling_Mean_6', 'Rolling_Std_3', 'Same_Month_Last_Year', 'Month_Sin', 'Month_Cos', 'Time_Index']


In [14]:
sales_transactions = dataframes["sales_transactions"]

print("Dataset Shape:", sales_transactions.shape)
print("Required Columns:")
print(sales_transactions[["Order_Date", "SKU", "Quantity"]].head())

print("\nDate Range:")
print(sales_transactions["Order_Date"].min())
print(sales_transactions["Order_Date"].max())

print("\nUnique SKUs:", sales_transactions["SKU"].nunique())
print("Missing Values:")
print(sales_transactions[["Order_Date", "SKU", "Quantity"]].isnull().sum())

Dataset Shape: (180000, 30)
Required Columns:
  Order_Date      SKU  Quantity
0 2024-05-11  CAT-059         1
1 2025-07-31  CAT-038         1
2 2026-05-23  CAT-028        24
3 2024-06-05  26115-7        48
4 2025-12-28  CAT-024         1

Date Range:
2023-10-01 00:00:00
2026-09-30 00:00:00

Unique SKUs: 59
Missing Values:
Order_Date    0
SKU           0
Quantity      0
dtype: int64


In [15]:
print("Model Features:")
for feature in model_artifact["features"]:
    print("-", feature)

print("\nModel Forecast Months:")
print(model_artifact["forecast_months"])

print("\nModel Metadata:")
print("Target:", model_artifact["target"])
print("Training Start:", model_artifact["data_start"])
print("Training End:", model_artifact["data_end"])

Model Features:
- SKU
- Lag_1
- Lag_2
- Lag_3
- Lag_6
- Lag_12
- Rolling_Mean_3
- Rolling_Mean_6
- Rolling_Std_3
- Same_Month_Last_Year
- Month_Sin
- Month_Cos
- Time_Index

Model Forecast Months:
['2026-10', '2026-11', '2026-12']

Model Metadata:
Target: Target_Quantity
Training Start: 2023-09
Training End: 2026-09


In [16]:
sales_transactions = dataframes["sales_transactions"].copy()

sales_transactions["Month"] = (
    sales_transactions["Order_Date"].dt.to_period("M")
)

monthly_sku_data = (
    sales_transactions
    .groupby(["SKU", "Month"], as_index=False)
    .agg(Quantity=("Quantity", "sum"))
    .sort_values(["SKU", "Month"])
    .reset_index(drop=True)
)

DATA_END = monthly_sku_data["Month"].max()

print("Monthly Data Shape:", monthly_sku_data.shape)
print("Unique SKUs:", monthly_sku_data["SKU"].nunique())
print("Data Start:", monthly_sku_data["Month"].min())
print("Data End:", DATA_END)
print("Missing Values:")
print(monthly_sku_data[["SKU", "Month", "Quantity"]].isnull().sum())

display(monthly_sku_data.head())

Monthly Data Shape: (2124, 3)
Unique SKUs: 59
Data Start: 2023-10
Data End: 2026-09
Missing Values:
SKU         0
Month       0
Quantity    0
dtype: int64


,SKU,Month,Quantity
0,26115-1,2023-10,379
1,26115-1,2023-11,233
2,26115-1,2023-12,448
3,26115-1,2024-01,287
4,26115-1,2024-02,499


In [18]:
expected_months = pd.period_range(
    start="2023-10",
    end="2026-09",
    freq="M"
)

months_per_sku = monthly_sku_data.groupby("SKU")["Month"].nunique()

print("Expected Months per SKU:", len(expected_months))
print("Minimum Months per SKU:", months_per_sku.min())
print("Maximum Months per SKU:", months_per_sku.max())
print("SKUs with Missing Months:", (months_per_sku != len(expected_months)).sum())

Expected Months per SKU: 36
Minimum Months per SKU: 36
Maximum Months per SKU: 36
SKUs with Missing Months: 0


In [19]:
# 1. Load the saved model and features

model = model_artifact["model"]
FEATURES = model_artifact["features"]

FORECAST_MONTHS = pd.period_range(
    start="2026-10",
    end="2026-12",
    freq="M"
)

DATA_END = monthly_sku_data["Month"].max()

if DATA_END != pd.Period("2026-09", freq="M"):
    raise ValueError(f"Unexpected data end: {DATA_END}")

# 2. Prepare monthly history for each SKU

history_by_sku = {}

for sku, group in monthly_sku_data.groupby("SKU", sort=False):
    group = group.sort_values("Month")
    history_by_sku[sku] = group["Quantity"].astype(float).tolist()

# 3. Generate recursive forecasts

forecast_records = []

for forecast_month in FORECAST_MONTHS:

    current_month_predictions = []

    for sku, history in history_by_sku.items():

        lag_1 = history[-1]
        lag_2 = history[-2]
        lag_3 = history[-3]
        lag_6 = history[-6]
        lag_12 = history[-12]

        rolling_mean_3 = np.mean(history[-3:])
        rolling_mean_6 = np.mean(history[-6:])
        rolling_std_3 = np.std(history[-3:], ddof=1)

        same_month_last_year = history[-12]

        origin_month = forecast_month - 1

        month_sin = np.sin(
            2 * np.pi * origin_month.month / 12
        )
        month_cos = np.cos(
            2 * np.pi * origin_month.month / 12
        )

        time_index = len(history) - 1

        input_row = pd.DataFrame([{
            "SKU": sku,
            "Lag_1": lag_1,
            "Lag_2": lag_2,
            "Lag_3": lag_3,
            "Lag_6": lag_6,
            "Lag_12": lag_12,
            "Rolling_Mean_3": rolling_mean_3,
            "Rolling_Mean_6": rolling_mean_6,
            "Rolling_Std_3": rolling_std_3,
            "Same_Month_Last_Year": same_month_last_year,
            "Month_Sin": month_sin,
            "Month_Cos": month_cos,
            "Time_Index": time_index
        }])

        predicted_quantity = max(
            float(model.predict(input_row[FEATURES])[0]),
            0
        )

        current_month_predictions.append({
            "SKU": sku,
            "Forecast_Month": str(forecast_month),
            "Predicted_Quantity": predicted_quantity
        })

    forecast_records.extend(current_month_predictions)

    # Feed predictions into history for the next month
    for record in current_month_predictions:
        history_by_sku[record["SKU"]].append(
            record["Predicted_Quantity"]
        )

# 4. Build forecast DataFrame

forecast_df = pd.DataFrame(forecast_records)

forecast_df = forecast_df.sort_values(
    ["SKU", "Forecast_Month"]
).reset_index(drop=True)

# 5. Summarize monthly forecasts

monthly_forecast = (
    forecast_df.groupby("Forecast_Month", as_index=False)
    .agg(
        Total_Predicted_Quantity=("Predicted_Quantity", "sum"),
        Number_of_SKUs=("SKU", "nunique")
    )
)

print("Forecast Shape:", forecast_df.shape)
print("Unique SKUs:", forecast_df["SKU"].nunique())
print("Forecast Months:", forecast_df["Forecast_Month"].unique())
print("Missing Values:", forecast_df.isnull().sum().sum())
print("Negative Forecasts:", (forecast_df["Predicted_Quantity"] < 0).sum())

display(monthly_forecast)
display(forecast_df.head(10))

Forecast Shape: (177, 3)
Unique SKUs: 59
Forecast Months: ['2026-10' '2026-11' '2026-12']
Missing Values: 0
Negative Forecasts: 0


,Forecast_Month,Total_Predicted_Quantity,Number_of_SKUs
0,2026-10,"37,685.19",59
1,2026-11,"43,427.43",59
2,2026-12,"54,585.63",59


,SKU,Forecast_Month,Predicted_Quantity
0,26115-1,2026-10,285.44
1,26115-1,2026-11,211.03
2,26115-1,2026-12,336.94
3,26115-2,2026-10,410.26
4,26115-2,2026-11,329.16
5,26115-2,2026-12,705.53
6,26115-3,2026-10,215.34
7,26115-3,2026-11,308.47
8,26115-3,2026-12,259.46
9,26115-4,2026-10,"1,094.42"


In [ ]:
# Aggregate the three-month forecast for each SKU

sku_forecast_summary = (
    forecast_df
    .groupby("SKU", as_index=False)
    .agg(
        Total_3M_Forecast=("Predicted_Quantity", "sum"),
        Monthly_Forecast_Mean=("Predicted_Quantity", "mean")
    )
)

# Merge forecast with the latest inventory

planning_df = sku_forecast_summary.merge(
    sku_inventory,
    on="SKU",
    how="left",
    validate="one_to_one"
)

print("Planning Dataset Shape:", planning_df.shape)
print("Unique SKUs:", planning_df["SKU"].nunique())
print("Missing Values After Merge:")
print(
    planning_df[
        ["Total_3M_Forecast", "Total_Closing_Stock",
         "Total_Safety_Stock", "Total_Reorder_Point"]
    ].isnull().sum()
)

display(planning_df.head()) # هو انا كدا بقى عند ما عملت التنبؤات ودمجتها مع المخزون الحالي، دلوقتي عندنا جدول بيحتوي على كل SKU مع توقعات الثلاث شهور القادمة، المخزون الحالي، المخزون الآمن، ونقطة إعادة الطلب. ده هيساعدنا في التخطيط للمخزون بشكل أفضل واتخاذ قرارات شراء أو إنتاج بناءً على التوقعات والمخزون الحالي.

Planning Dataset Shape: (59, 12)
Unique SKUs: 59
Missing Values After Merge:
Total_3M_Forecast      0
Total_Closing_Stock    0
Total_Safety_Stock     0
Total_Reorder_Point    0
dtype: int64


,SKU,Total_3M_Forecast,Monthly_Forecast_Mean,Product_ID,Product_Name,Total_Closing_Stock,Total_Safety_Stock,Total_Reorder_Point,Avg_Daily_Sales_90D,Supplier_Lead_Time_Days,Total_Stockout_Days,Locations_Count
0,26115-1,833.42,277.81,P006,Summer children's boys-Sonic,288,349,885,38.85,21,0,3
1,26115-2,"1,444.94",481.65,P005,Summer children's boys-Black and orange,588,714,1812,55.70,21,0,3
2,26115-3,783.27,261.09,P007,Summer children's boys-Black and turquoise,719,88,223,26.95,5,0,3
3,26115-4,"3,455.30","1,151.77",P008,Summer children's boys-SpongeBob,1061,657,1667,100.66,14,0,3
4,26115-5,"1,184.79",394.93,P003,Summer children's girls-Hello kitty Rose & tur...,583,169,427,49.23,7,0,3


In [24]:

# ==================================================
# Inventory Risk Analysis
# ==================================================

# 1. Calculate forecast horizon
forecast_start = pd.Period("2026-10", freq="M").start_time
forecast_end = pd.Period("2026-12", freq="M").end_time

forecast_horizon_days = (
    forecast_end.date() - forecast_start.date()
).days + 1

# 2. Calculate average daily forecast demand
planning_df["Forecast_Daily_Demand"] = (
    planning_df["Total_3M_Forecast"] / forecast_horizon_days
)

# 3. Estimate demand during supplier lead time
planning_df["Lead_Time_Demand"] = (
    planning_df["Forecast_Daily_Demand"]
    * planning_df["Supplier_Lead_Time_Days"]
)

# 4. Calculate inventory coverage gap
planning_df["Lead_Time_Coverage_Gap"] = (
    planning_df["Total_Closing_Stock"]
    - planning_df["Lead_Time_Demand"]
)

planning_df["Below_Reorder_Point"] = (
    planning_df["Total_Closing_Stock"]
    < planning_df["Total_Reorder_Point"]
)

planning_df["Lead_Time_Shortage_Risk"] = (
    planning_df["Total_Closing_Stock"]
    < planning_df["Lead_Time_Demand"]
)

# 5. Classify inventory risk
planning_df["Inventory_Risk"] = np.select(
    [
        planning_df["Lead_Time_Shortage_Risk"],
        planning_df["Below_Reorder_Point"]
    ],
    [
        "Lead-Time Shortage Risk",
        "Below Reorder Point"
    ],
    default="Stock Above Thresholds"
)

# 6. Calculate inventory shortfall
planning_df["Lead_Time_Shortfall"] = (
    planning_df["Lead_Time_Demand"]
    - planning_df["Total_Closing_Stock"]
).clip(lower=0)

# 7. Display risk distribution
print("Inventory Risk Distribution")

risk_summary = (
    planning_df["Inventory_Risk"]
    .value_counts()
    .rename_axis("Inventory_Risk")
    .reset_index(name="SKU_Count")
)

display(risk_summary)

# 8. Display inventory planning table
planning_df = planning_df.sort_values(
    ["Lead_Time_Shortage_Risk", "Below_Reorder_Point",
     "Lead_Time_Shortfall"],
    ascending=[False, False, False]
).reset_index(drop=True)

display(
    planning_df[
        [
            "SKU",
            "Total_Closing_Stock",
            "Total_3M_Forecast",
            "Forecast_Daily_Demand",
            "Supplier_Lead_Time_Days",
            "Lead_Time_Demand",
            "Lead_Time_Coverage_Gap",
            "Total_Reorder_Point",
            "Below_Reorder_Point",
            "Lead_Time_Shortfall",
            "Inventory_Risk"
        ]
    ]
)


Inventory Risk Distribution


,Inventory_Risk,SKU_Count
0,Stock Above Thresholds,33
1,Below Reorder Point,17
2,Lead-Time Shortage Risk,9


,SKU,Total_Closing_Stock,Total_3M_Forecast,Forecast_Daily_Demand,Supplier_Lead_Time_Days,Lead_Time_Demand,Lead_Time_Coverage_Gap,Total_Reorder_Point,Below_Reorder_Point,Lead_Time_Shortfall,Inventory_Risk
0,CAT-039,441,"7,222.08",78.50,21,"1,648.52","-1,207.52",455,True,"1,207.52",Lead-Time Shortage Risk
1,26115-6,5,"1,159.69",12.61,21,264.71,-259.71,1056,True,259.71,Lead-Time Shortage Risk
2,26115-8,597,"3,626.48",39.42,21,827.78,-230.78,1382,True,230.78,Lead-Time Shortage Risk
3,CAT-033,419,"8,006.73",87.03,14,"1,218.42",-799.42,153,False,799.42,Lead-Time Shortage Risk
4,CAT-044,370,"6,590.15",71.63,14,"1,002.85",-632.85,215,False,632.85,Lead-Time Shortage Risk
5,CAT-045,583,"6,575.32",71.47,14,"1,000.59",-417.59,250,False,417.59,Lead-Time Shortage Risk
6,CAT-035,417,"3,397.65",36.93,21,775.55,-358.55,107,False,358.55,Lead-Time Shortage Risk
7,CAT-042,296,"2,571.26",27.95,21,586.92,-290.92,100,False,290.92,Lead-Time Shortage Risk
8,CAT-036,574,"3,348.34",36.40,21,764.30,-190.30,130,False,190.30,Lead-Time Shortage Risk
9,26115-1,288,833.42,9.06,21,190.24,97.76,885,True,0.00,Below Reorder Point


In [25]:

# Prepare the inventory simulation data

simulation_df = forecast_df.merge(
    sku_inventory[["SKU", "Total_Closing_Stock"]],
    on="SKU",
    how="left",
    validate="many_to_one"
)

simulation_df["Forecast_Month"] = simulation_df["Forecast_Month"].astype(str)

simulation_df = simulation_df.sort_values(
    ["SKU", "Forecast_Month"]
).reset_index(drop=True)


# Simulate inventory month by month

simulation_records = []

for sku, group in simulation_df.groupby("SKU", sort=False):

    opening_stock = float(group["Total_Closing_Stock"].iloc[0])

    for row in group.itertuples(index=False):

        forecast_demand = max(float(row.Predicted_Quantity), 0.0)

        available_stock = opening_stock

        fulfilled_demand = min(available_stock, forecast_demand)

        unmet_demand = max(
            forecast_demand - available_stock,
            0.0
        )

        closing_stock = max(
            available_stock - forecast_demand,
            0.0
        )

        simulation_records.append({
            "SKU": sku,
            "Forecast_Month": row.Forecast_Month,
            "Opening_Stock": opening_stock,
            "Forecast_Demand": forecast_demand,
            "Available_Stock": available_stock,
            "Fulfilled_Demand": fulfilled_demand,
            "Unmet_Demand": unmet_demand,
            "Closing_Stock": closing_stock,
            "Stockout_Flag": unmet_demand > 0
        })

        # Carry the remaining stock into the next month
        opening_stock = closing_stock


inventory_simulation_df = pd.DataFrame(simulation_records)


# Summarize the simulation by month

monthly_simulation_summary = (
    inventory_simulation_df
    .groupby("Forecast_Month", as_index=False)
    .agg(
        Opening_Stock=("Opening_Stock", "sum"),
        Forecast_Demand=("Forecast_Demand", "sum"),
        Fulfilled_Demand=("Fulfilled_Demand", "sum"),
        Unmet_Demand=("Unmet_Demand", "sum"),
        Closing_Stock=("Closing_Stock", "sum"),
        SKUs_With_Stockout=("Stockout_Flag", "sum")
    )
)

display(monthly_simulation_summary)


,Forecast_Month,Opening_Stock,Forecast_Demand,Fulfilled_Demand,Unmet_Demand,Closing_Stock,SKUs_With_Stockout
0,2026-10,"77,867.00","37,685.19","26,319.30","11,365.89","51,547.70",18
1,2026-11,"51,547.70","43,427.43","10,649.54","32,777.89","40,898.16",37
2,2026-12,"40,898.16","54,585.63","5,879.06","48,706.57","35,019.10",50


In [27]:

october_check = inventory_simulation_df[
    inventory_simulation_df["Forecast_Month"] == "2026-10"
].copy()

october_check["Expected_Fulfilled"] = october_check[
    ["Opening_Stock", "Forecast_Demand"]
].min(axis=1)

october_check["Expected_Unmet"] = (
    october_check["Forecast_Demand"]
    - october_check["Expected_Fulfilled"]
).clip(lower=0)

october_check["Expected_Closing"] = (
    october_check["Opening_Stock"]
    - october_check["Expected_Fulfilled"]
)

october_check["Fulfilled_Mismatch"] = (
    october_check["Fulfilled_Demand"]
    - october_check["Expected_Fulfilled"]
).abs() > 1e-6

october_check["Closing_Mismatch"] = (
    october_check["Closing_Stock"]
    - october_check["Expected_Closing"]
).abs() > 1e-6

print("Rows:", len(october_check))
print("Fulfilled mismatches:", october_check["Fulfilled_Mismatch"].sum())
print("Closing mismatches:", october_check["Closing_Mismatch"].sum())

display(
    october_check.loc[
        october_check["Fulfilled_Mismatch"] | october_check["Closing_Mismatch"],
        [
            "SKU",
            "Opening_Stock",
            "Forecast_Demand",
            "Fulfilled_Demand",
            "Expected_Fulfilled",
            "Unmet_Demand",
            "Closing_Stock",
            "Expected_Closing"
        ]
    ].head(10)
)


Rows: 59
Fulfilled mismatches: 0
Closing mismatches: 0


,SKU,Opening_Stock,Forecast_Demand,Fulfilled_Demand,Expected_Fulfilled,Unmet_Demand,Closing_Stock,Expected_Closing


In [28]:

# Analyze surplus and shortage for each SKU

inventory_simulation_df["Surplus"] = (
    inventory_simulation_df["Closing_Stock"]
)

inventory_simulation_df["Shortage"] = (
    inventory_simulation_df["Unmet_Demand"]
)

inventory_simulation_df["Inventory_Status"] = np.select(
    [
        inventory_simulation_df["Shortage"] > 0,
        inventory_simulation_df["Surplus"] > 0
    ],
    [
        "Shortage",
        "Surplus"
    ],
    default="Balanced"
)

display(
    inventory_simulation_df[
        [
            "SKU",
            "Forecast_Month",
            "Opening_Stock",
            "Forecast_Demand",
            "Fulfilled_Demand",
            "Unmet_Demand",
            "Closing_Stock",
            "Inventory_Status"
        ]
    ].sort_values(
        ["Forecast_Month", "Unmet_Demand"],
        ascending=[True, False]
    ).reset_index(drop=True)
)


,SKU,Forecast_Month,Opening_Stock,Forecast_Demand,Fulfilled_Demand,Unmet_Demand,Closing_Stock,Inventory_Status
0,CAT-033,2026-10,419.00,"2,210.45",419.00,"1,791.45",0.00,Shortage
1,CAT-039,2026-10,441.00,"1,975.97",441.00,"1,534.97",0.00,Shortage
2,CAT-044,2026-10,370.00,"1,599.06",370.00,"1,229.06",0.00,Shortage
3,CAT-043,2026-10,722.00,"1,757.68",722.00,"1,035.68",0.00,Shortage
4,CAT-038,2026-10,588.00,"1,603.58",588.00,"1,015.58",0.00,Shortage
...,...,...,...,...,...,...,...,...
172,CAT-052,2026-12,431.38,191.36,191.36,0.00,240.01,Surplus
173,CAT-054,2026-12,"9,176.80",357.51,357.51,0.00,"8,819.29",Surplus
174,CAT-056,2026-12,698.56,556.17,556.17,0.00,142.39,Surplus
175,CAT-057,2026-12,298.09,239.78,239.78,0.00,58.31,Surplus


In [29]:

sku_inventory_summary = (
    inventory_simulation_df
    .groupby("SKU", as_index=False)
    .agg(
        Total_Forecast_Demand=("Forecast_Demand", "sum"),
        Total_Fulfilled_Demand=("Fulfilled_Demand", "sum"),
        Total_Unmet_Demand=("Unmet_Demand", "sum"),
        Final_Closing_Stock=("Closing_Stock", "last"),
        Months_With_Shortage=("Stockout_Flag", "sum")
    )
)

sku_inventory_summary["Planning_Status"] = np.select(
    [
        sku_inventory_summary["Total_Unmet_Demand"] > 0,
        sku_inventory_summary["Final_Closing_Stock"] > 0
    ],
    [
        "Production Review Needed",
        "Stock Remaining"
    ],
    default="No Remaining Stock"
)

sku_inventory_summary = sku_inventory_summary.sort_values(
    ["Total_Unmet_Demand", "Final_Closing_Stock"],
    ascending=[False, False]
).reset_index(drop=True)

display(sku_inventory_summary)


,SKU,Total_Forecast_Demand,Total_Fulfilled_Demand,Total_Unmet_Demand,Final_Closing_Stock,Months_With_Shortage,Planning_Status
0,CAT-037,"10,335.61","2,665.00","7,670.61",0.00,3,Production Review Needed
1,CAT-033,"8,006.73",419.00,"7,587.73",0.00,3,Production Review Needed
2,CAT-039,"7,222.08",441.00,"6,781.08",0.00,3,Production Review Needed
3,CAT-038,"7,029.01",588.00,"6,441.01",0.00,3,Production Review Needed
4,CAT-044,"6,590.15",370.00,"6,220.15",0.00,3,Production Review Needed
5,CAT-045,"6,575.32",583.00,"5,992.32",0.00,3,Production Review Needed
6,CAT-032,"7,072.88","1,205.00","5,867.88",0.00,3,Production Review Needed
7,CAT-043,"6,329.21",722.00,"5,607.21",0.00,3,Production Review Needed
8,26115-8,"3,626.48",597.00,"3,029.48",0.00,3,Production Review Needed
9,CAT-035,"3,397.65",417.00,"2,980.65",0.00,3,Production Review Needed


In [30]:

# Add product details to the inventory planning summary

product_master = dataframes["product_master"].copy()

sku_production_review = sku_inventory_summary.merge(
    product_master,
    on="SKU",
    how="left",
    validate="one_to_one"
)

sku_production_review = sku_production_review.sort_values(
    "Total_Unmet_Demand",
    ascending=False
).reset_index(drop=True)

print("Total SKUs:", sku_production_review["SKU"].nunique())
print(
    "SKUs requiring production review:",
    (sku_production_review["Total_Unmet_Demand"] > 0).sum()
)

display(
    sku_production_review[
        [
            "SKU",
            "Product_Name",
            "Total_Forecast_Demand",
            "Total_Fulfilled_Demand",
            "Total_Unmet_Demand",
            "Final_Closing_Stock",
            "Months_With_Shortage",
            "Planning_Status"
        ]
    ]
)


Total SKUs: 59
SKUs requiring production review: 50


,SKU,Product_Name,Total_Forecast_Demand,Total_Fulfilled_Demand,Total_Unmet_Demand,Final_Closing_Stock,Months_With_Shortage,Planning_Status
0,CAT-037,Girls' sweatshirt-Flower,"10,335.61","2,665.00","7,670.61",0.00,3,Production Review Needed
1,CAT-033,Boys' sweatshirt-LEO,"8,006.73",419.00,"7,587.73",0.00,3,Production Review Needed
2,CAT-039,Boys' Autumn Pajamas-Spider Man,"7,222.08",441.00,"6,781.08",0.00,3,Production Review Needed
3,CAT-038,Boys' Autumn Pajamas-Mickey Mouse,"7,029.01",588.00,"6,441.01",0.00,3,Production Review Needed
4,CAT-044,Girls' autumn pajamas-Ice Cream,"6,590.15",370.00,"6,220.15",0.00,3,Production Review Needed
5,CAT-045,Girls' autumn pajamas-Stitch,"6,575.32",583.00,"5,992.32",0.00,3,Production Review Needed
6,CAT-032,Boys' sweatshirt-blue,"7,072.88","1,205.00","5,867.88",0.00,3,Production Review Needed
7,CAT-043,Girls' autumn pajamas-Snow White,"6,329.21",722.00,"5,607.21",0.00,3,Production Review Needed
8,26115-8,Summer children's girls-Happy,"3,626.48",597.00,"3,029.48",0.00,3,Production Review Needed
9,CAT-035,Girls' sweatshirt-Fuchsia,"3,397.65",417.00,"2,980.65",0.00,3,Production Review Needed


In [31]:

# Create a monthly shortage table for production planning

monthly_shortage_df = inventory_simulation_df[
    [
        "SKU",
        "Forecast_Month",
        "Forecast_Demand",
        "Opening_Stock",
        "Fulfilled_Demand",
        "Unmet_Demand",
        "Closing_Stock"
    ]
].copy()

monthly_shortage_df = monthly_shortage_df.sort_values(
    ["Forecast_Month", "Unmet_Demand"],
    ascending=[True, False]
).reset_index(drop=True)

monthly_shortage_summary = (
    monthly_shortage_df
    .groupby("Forecast_Month", as_index=False)
    .agg(
        Total_Demand=("Forecast_Demand", "sum"),
        Total_Shortage=("Unmet_Demand", "sum"),
        SKUs_Needing_Replenishment=(
            "Unmet_Demand",
            lambda values: (values > 0).sum()
        )
    )
)

display(monthly_shortage_summary)


,Forecast_Month,Total_Demand,Total_Shortage,SKUs_Needing_Replenishment
0,2026-10,"37,685.19","11,365.89",18
1,2026-11,"43,427.43","32,777.89",37
2,2026-12,"54,585.63","48,706.57",50


In [32]:

from pathlib import Path

import pandas as pd
import numpy as np

# Create output directory
OUTPUT_DIR = Path("streamlit_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# 1. Product catalog
product_master = dataframes["product_master"].copy()

product_catalog = (
    product_master[
        ["SKU", "Product_Name"]
    ]
    .drop_duplicates(subset=["SKU"])
    .sort_values("SKU")
    .reset_index(drop=True)
)

product_catalog.to_csv(
    OUTPUT_DIR / "Product_Catalog.csv",
    index=False
)


# 2. Monthly forecasts
monthly_forecasts = forecast_df[
    ["SKU", "Forecast_Month", "Predicted_Quantity"]
].copy()

monthly_forecasts["Forecast_Month"] = (
    monthly_forecasts["Forecast_Month"].astype(str)
)

monthly_forecasts = monthly_forecasts.rename(
    columns={
        "Predicted_Quantity": "Forecast_Demand"
    }
)

monthly_forecasts.to_csv(
    OUTPUT_DIR / "Monthly_Forecasts.csv",
    index=False
)


# 3. Inventory planning recommendations
planning_inventory = sku_inventory[
    ["SKU", "Total_Closing_Stock", "Total_Safety_Stock"]
].copy()

inventory_planning = monthly_forecasts.merge(
    planning_inventory,
    on="SKU",
    how="left",
    validate="many_to_one"
).merge(
    product_catalog,
    on="SKU",
    how="left",
    validate="many_to_one"
)

inventory_planning = inventory_planning.rename(
    columns={
        "Total_Closing_Stock": "Current_Stock",
        "Total_Safety_Stock": "Safety_Stock"
    }
)

inventory_planning = inventory_planning.sort_values(
    ["SKU", "Forecast_Month"]
).reset_index(drop=True)

inventory_planning["Augmented_Demand"] = (
    inventory_planning["Forecast_Demand"]
    + inventory_planning["Safety_Stock"]
)

inventory_planning["Demand_Shortage"] = (
    inventory_planning["Augmented_Demand"]
    - inventory_planning["Current_Stock"]
).clip(lower=0)

inventory_planning["Recommended_Production"] = (
    inventory_planning["Demand_Shortage"]
)

inventory_planning.to_csv(
    OUTPUT_DIR / "Inventory_Planning.csv",
    index=False
)


# Verify saved files
for file_path in sorted(OUTPUT_DIR.glob("*.csv")):
    saved_df = pd.read_csv(file_path)

    print(
        f"{file_path.name}: "
        f"{saved_df.shape[0]} rows, "
        f"{saved_df.shape[1]} columns"
    )


Inventory_Planning.csv: 177 rows, 9 columns
Monthly_Forecasts.csv: 177 rows, 3 columns
Product_Catalog.csv: 59 rows, 2 columns
